In [2]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")

    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/Shareddrives/Progetti/Progetto_MLDM')

    GIT_BRANCH = 'zone-test'

    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd

    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)

    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)

else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")

    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR = DATA_DIR / 'models'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")



Ambiente Colab rilevato. Inizializzazione in corso...
branch 'zone-test' set up to track 'origin/zone-test'.
Reset branch 'zone-test'
Your branch is up to date with 'origin/zone-test'.
HEAD is now at b367b99 fix errore
Setup ambiente Colab completato! Branch attivo su Colab:  zone-test
✅ Collegamento ai dati riuscito! Cartella raw: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw


# 04 - Addestramento modelli
- Addestramento dei modelli Random Forest, ExtraTrees e XGBoost (con ricerca dei migliori iperparametri)
- Dataset di addestramento costruito sui punti dell'area della Capitanata
- Feature importance: confronto tra modelli
- Feature pruning sul modello ExtraTrees

#### Caricamento del dataset e selezione delle feature predittive

In [3]:
import json
import pandas as pd

from src.feature_engineering import FEATURE_LIST

dataset_path = PROCESSED_DIR / 'dataset' / 'dataset.parquet'
legend_path = PROCESSED_DIR / 'legend.json'

dataset = pd.read_parquet(dataset_path)

# esclusione delle features non utili ai fini dell'addestramento
features = [f for f in dataset.columns if f in FEATURE_LIST]

X = dataset[features]
y = dataset["Ground_Truth"].astype(int)

# caricamento legenda delle colture
legend = {}
if legend_path.exists():
    with open(legend_path, "r", encoding="utf-8") as f:
        legend = json.load(f)

print(f"[Capitanata]\nDataset di addestramento:")
print(f"  • Records: {X.shape[0]}")
print(f"  • Features: {X.shape[1]}")
print(f"  • Classi: {y.nunique()}")

[Capitanata]
Dataset di addestramento:
  • Records: 22500
  • Features: 51
  • Classi: 17


#### Addestramento - RandomForest

In [ ]:
import time
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

extra_trees_dir = MODELS_DIR / "random_forest"
extra_trees_dir.mkdir(parents=True, exist_ok=True)
model_path = extra_trees_dir / "random_forest_crop_model.joblib"

if model_path.exists():
    print(f"Modello RandomForestClassifier trovato su disco: {model_path.name}")
    print("Caricamento del modello completato!")
    payload = joblib.load(model_path)
    base_model = payload["model"]
    best_params = payload.get("best_params")
    best_cv_f1 = payload.get("best_cv_f1_macro")
    cv_results = payload.get("cv_results")
    print(f"  • Parametri migliori: {best_params}")
    print(f"  • F1-Macro (CV):      {best_cv_f1:.4f}")
else:
    print("Modello non trovato. Inizio addestramento RandomForestClassifier (GridSearchCV)...")
    base_model = RandomForestClassifier(random_state=42, max_features="sqrt")
    param_grid = {
        "n_estimators": [100, 300, 500, 700],
        "max_depth": [None, 25],
        "min_samples_split": [2],
        "min_samples_leaf": [1],
        "class_weight": ["balanced_subsample"],
    }
    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring=["accuracy", "f1_macro"],
        refit="f1_macro",
        cv=5,
        n_jobs=-1,
        verbose=1,
    )

    start_time = time.time()
    grid_search.fit(X, y)
    fit_time = time.time() - start_time
    print(f"Addestramento completato in {fit_time:.2f} secondi.")

    base_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    best_cv_f1 = grid_search.best_score_

    # salvataggio del modello su disco
    payload = {
        "model": base_model,
        "best_params": best_params,
        "best_cv_f1_macro": best_cv_f1,
        "features": list(X.columns),
        "n_features": X.shape[1],
        "threshold": None,
        "training_time_seconds": fit_time,
        "cv_results": grid_search.cv_results_,
    }
    joblib.dump(payload, model_path)
    print(f"Modello salvato con successo in: {model_path.name}")

Modello RandomForestClassifier trovato su disco: random_forest_crop_model.joblib
Caricamento del modello completato!
  • Parametri migliori: {'class_weight': 'balanced_subsample', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 700}
  • F1-Macro (CV):      0.5618


#### Addestramento - ExtraTrees

In [ ]:
import time
import joblib
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV

extra_trees_dir = MODELS_DIR / "extra_trees"
extra_trees_dir.mkdir(parents=True, exist_ok=True)
model_path = extra_trees_dir / "extra_trees_crop_model.joblib"

if model_path.exists():
    print(f"Modello ExtraTreesClassifier trovato su disco: {model_path.name}")
    print("Caricamento del modello completato!")
    payload = joblib.load(model_path)
    base_model = payload["model"]
    best_params = payload.get("best_params")
    best_cv_f1 = payload.get("best_cv_f1_macro")
    cv_results = payload.get("cv_results")
    print(f"  • Parametri migliori: {best_params}")
    print(f"  • F1-Macro (CV):      {best_cv_f1:.4f}")
else:
    print("Modello non trovato. Inizio addestramento ExtraTreesClassifier (GridSearchCV)...")
    base_model = ExtraTreesClassifier(random_state=42, max_features="sqrt")
    param_grid = {
        "n_estimators": [100, 300, 500, 700],
        "max_depth": [None, 25],
        "min_samples_split": [2],
        "min_samples_leaf": [1],
        "class_weight": ["balanced_subsample"],
    }
    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring=["accuracy", "f1_macro"],
        refit="f1_macro",
        cv=5,
        n_jobs=-1,
        verbose=1,
    )

    start_time = time.time()
    grid_search.fit(X, y)
    fit_time = time.time() - start_time
    print(f"Addestramento completato in {fit_time:.2f} secondi.")

    base_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    best_cv_f1 = grid_search.best_score_

    # salvataggio del modello su disco
    payload = {
        "model": base_model,
        "best_params": best_params,
        "best_cv_f1_macro": best_cv_f1,
        "features": list(X.columns),
        "n_features": X.shape[1],
        "threshold": None,
        "training_time_seconds": fit_time,
        "cv_results": grid_search.cv_results_,
    }
    joblib.dump(payload, model_path)
    print(f"Modello salvato con successo in: {model_path.name}")

Modello ExtraTreesClassifier trovato su disco: extra_trees_crop_model.joblib
Caricamento del modello completato!
  • Parametri migliori: {'class_weight': 'balanced_subsample', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 700}
  • F1-Macro (CV):      0.5648


#### Addestramento - XGBoost

In [ ]:
import time
import joblib
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

xgb_dir = MODELS_DIR / "xgboost"
xgb_dir.mkdir(parents=True, exist_ok=True)
model_path = xgb_dir / "xgboost_crop_model.joblib"

if model_path.exists():
    print(f"Modello XGBClassifier trovato su disco: {model_path.name}")
    print("Caricamento del modello completato!")
    payload = joblib.load(model_path)
    base_model = payload["model"]
    best_params = payload.get("best_params")
    best_cv_f1 = payload.get("best_cv_f1_macro")
    cv_results = payload.get("cv_results")
    print(f"  • Parametri migliori: {best_params}")
    print(f"  • F1-Macro (CV):      {best_cv_f1:.4f}")
else:
    print("Modello non trovato. Inizio addestramento XGBClassifier (GridSearchCV)...")
    base_model = XGBClassifier(
        random_state=42,
        colsample_bytree=0.8,
        subsample=0.8,
        eval_metric="mlogloss",
    )
    param_grid = {
        "n_estimators": [200, 400],
        "learning_rate": [0.05, 0.1],
        "max_depth": [6, 8],
        "min_child_weight": [1, 3],
    }
    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring=["accuracy", "f1_macro"],
        refit="f1_macro",
        cv=5,
        n_jobs=-1,
        verbose=1,
    )

    # codifica dei codici delle colture
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    start_time = time.time()
    grid_search.fit(X, y_encoded)
    fit_time = time.time() - start_time
    print(f"Addestramento completato in {fit_time:.2f} secondi.")

    base_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    best_cv_f1 = grid_search.best_score_

    # salvataggio del modello su disco
    payload = {
        "model": base_model,
        "best_params": best_params,
        "label_encoder": label_encoder,
        "best_cv_f1_macro": best_cv_f1,
        "features": list(X.columns),
        "n_features": X.shape[1],
        "threshold": None,
        "training_time_seconds": fit_time,
        "cv_results": grid_search.cv_results_,
    }
    joblib.dump(payload, model_path)
    print(f"Modello salvato con successo in: {model_path.name}")


Modello XGBClassifier trovato su disco: xgboost_crop_model.joblib
Caricamento del modello completato!
  • Parametri migliori: {'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 400}
  • F1-Macro (CV):      0.5815


#### Feature Importance - Confronto

In [ ]:
import pandas as pd
import joblib
from IPython.display import HTML, display

for model_dir in MODELS_DIR.iterdir():
    if not model_dir.is_dir():
        continue

    model_path = model_dir / f"{model_dir.name}_crop_model.joblib"

    if not model_path.exists():
        print("File non trovato in: {model_dir.name}")
        continue

    print(f"Caricamento modello da: {model_path.name}")
    payload = joblib.load(model_path)
    base_model = payload["model"]
    features = payload["features"]

    NUM_FEATURES = 10

    feature_importance_df = pd.DataFrame({
        "feature": features,
        "importance": base_model.feature_importances_
    }).sort_values(by="importance", ascending=False)

    feature_importance_df["importance"] = feature_importance_df["importance"].round(4)

    top_features_df = feature_importance_df.head(NUM_FEATURES)
    bottom_features_df = feature_importance_df.tail(NUM_FEATURES).sort_values(by="importance", ascending=True)

    comparison_df = pd.DataFrame({
        f"Top {NUM_FEATURES} Feature": top_features_df["feature"].values,
        "Top Importance": top_features_df["importance"].values,
        f"Bottom {NUM_FEATURES} Feature": bottom_features_df["feature"].values,
        "Bottom Importance": bottom_features_df["importance"].values,
    })

    print(f"\n[{model_dir.name}]")
    display(HTML(comparison_df.to_html(index=False)))

Caricamento modello da: random_forest_crop_model.joblib

[random_forest]


Top 10 Feature,Top Importance,Bottom 10 Feature,Bottom Importance
NDVI_diff_lug_apr,0.0363,Peak_Month,0.0070
NDVI_min,0.0314,Active_Months_Count,0.0087
NDVI_08,0.0295,Rosso_B04,0.0123
NDWI_08,0.0292,Verde_B03,0.0133
NDVI_04,0.0290,NDWI_mean,0.0134
NDVI_03,0.0271,Blu_B02,0.0135
NDWI_04,0.0269,NDVI_max,0.0143
NDWI_07,0.0260,NDVI_01,0.0146
NDVI_05,0.0252,NDVI_AUC,0.0153
NDVI_diff_set_lug,0.0248,NDWI_01,0.0154


Caricamento modello da: extra_trees_crop_model.joblib

[extra_trees]


Top 10 Feature,Top Importance,Bottom 10 Feature,Bottom Importance
NDVI_diff_lug_apr,0.0299,Rosso_B04,0.0138
NDVI_08,0.0278,Blu_B02,0.0145
NDVI_04,0.0264,Verde_B03,0.0146
NDWI_08,0.0252,Orzo_Wheat_Ratio,0.0150
NDWI_04,0.0251,NDWI_mean,0.0154
NDVI_05,0.0251,SWIR_NIR_ratio,0.0157
NDVI_03,0.0249,NDWI_01,0.0161
NDVI_07,0.0242,NDVI_01,0.0161
NDWI_07,0.0241,NDVI_max,0.0163
NDVI_min,0.0236,SWIR2_B12,0.0164


Caricamento modello da: xgboost_crop_model.joblib

[xgboost]


Top 10 Feature,Top Importance,Bottom 10 Feature,Bottom Importance
NDVI_diff_lug_apr,0.0760,Verde_B03,0.0101
Active_Months_Count,0.0667,Rosso_B04,0.0103
NDVI_min,0.0498,Blu_B02,0.0112
NDVI_08,0.0296,NDVI_winter,0.0116
NDWI_05,0.0290,NDWI_01,0.0122
NDVI_diff_set_lug,0.0251,Greenup_Mar_Feb,0.0122
NDWI_04,0.0249,NDVI_01,0.0125
NDVI_07,0.0225,NDVI_mean,0.0126
NDVI_09,0.0215,NIR_B08,0.0128
NDVI_05,0.0214,NDWI_diff_lug_gen,0.0129


#### Feature Pruning sul modello ExtraTrees

In [ ]:
import time
import joblib
import pandas as pd
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV

model_path = MODELS_DIR / "extra_trees" / f"extra_trees_crop_model.joblib"
payload = joblib.load(model_path)
base_model = payload["model"]
n_features = payload["n_features"]
best_params = payload.get("best_params", {})

# inizializza il confronto con i dati del modello base (tutte le feature) già calcolati
results_comparison = [{
    "Strategy": "All features",
    "Threshold": "None",
    "N_Feature": n_features,
    "%_Feature": 100.0,
    "Macro_F1": round(payload.get("best_cv_f1_macro", 0), 4),
    "Fit_time (s)": f"{payload.get('training_time_seconds', 0):.1f}",
}]

pruning_strategies = {
    "Mean threshold": "mean",
    "Median threshold": "median",
    "Scaling factor (0.75*mean)": "0.75*mean"
}

total_features = n_features

# ciclo di potatura e valutazione
for i, (strategy_name, th) in enumerate(pruning_strategies.items(), 1):
    selector = SelectFromModel(estimator=base_model, threshold=th, prefit=True)
    mask = selector.get_support()
    selected_features = list(X.columns[mask])
    num_selected = len(selected_features)
    perc_selected = round((num_selected / total_features) * 100, 1)

    file_suffix = th.replace("*", "_")
    file_path = MODELS_DIR / "extra_trees" / f"extra_trees_pruned_{file_suffix}.joblib"

    if file_path.exists():
        payload_pruned = joblib.load(file_path)
        macro_f1 = payload_pruned.get("best_cv_f1_macro", 0)
        fit_time_str = f"{payload_pruned.get('training_time_seconds', 0):.1f}"
        print(f"[{i}/3] {strategy_name} | Caricato da disco ({num_selected} feature)")
    else:
        print(f"[{i}/3] {strategy_name} | Addestramento ({num_selected} feature)...")

        model = ExtraTreesClassifier(random_state=42, max_features="sqrt")

        param_grid = {
            "n_estimators": [100, 300, 500, 700],
            "max_depth": [None, 25],
            "min_samples_split": [2],
            "min_samples_leaf": [1],
            "class_weight": ["balanced_subsample"],
        }

        grid_search_pruned = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            scoring=["accuracy", "f1_macro"],
            refit="f1_macro",
            cv=5,
            n_jobs=2,
            verbose=1,
        )

        start_time = time.time()
        grid_search_pruned.fit(X[selected_features], y)
        fit_time = time.time() - start_time
        fit_time_str = f"{fit_time:.1f}"

        best_model_pruned = grid_search_pruned.best_estimator_
        best_params_pruned = grid_search_pruned.best_params_
        macro_f1 = grid_search_pruned.best_score_

        print(f"[soglia: {strategy_name}] Addestramento completato in {fit_time_str}s")

        # salvataggio del modello su disco
        joblib.dump({
            "model": best_model_pruned,
            "best_params": best_params_pruned,
            "best_cv_f1_macro": macro_f1,
            "features": selected_features,
            "n_features": num_selected,
            "threshold": th,
            "training_time_seconds": fit_time,
            "cv_results": grid_search_pruned.cv_results_,
        }, file_path)
        print(f"ExtraTrees salvato in: {file_path.name}")

        del grid_search_pruned, model
        gc.collect()

    results_comparison.append({
        "Strategy": strategy_name,
        "Threshold": th,
        "N_Feature": num_selected,
        "%_Feature": perc_selected,
        "Macro_F1": round(macro_f1, 4),
        "Fit_time (s)": fit_time_str
    })

df_comparison = pd.DataFrame(results_comparison)
print("\nAddestramenti completati con successo!")
display(df_comparison.style.hide(axis="index"))

[1/3] Mean threshold | Caricato da disco (20 feature)
[2/3] Median threshold | Addestramento (26 feature)...
Fitting 5 folds for each of 8 candidates, totalling 40 fits
